# Econometric Analysis of Education and Income

This notebook performs the primary econometric analysis on the cleaned Australian census data, examining the relationship between education and income.

## Import Libraries and Configure Paths

In [23]:
from pathlib import Path
import pandas as pd
import numpy as np
import statsmodels.api as sm
from IPython.display import Markdown, display

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)

# --- Locate project root by finding the data/clean folder ---
cwd = Path.cwd().resolve()
candidate_roots = [cwd, *cwd.parents]

project_root = next(
    (p for p in candidate_roots if (p / 'data' / 'clean').exists()),
    None
)

if project_root is None:
    raise FileNotFoundError("Could not locate project root containing data/clean/")

clean_dir = project_root / 'data' / 'clean'

# --- Find the merged file automatically ---
merged_files = list(clean_dir.glob("*merged*.csv"))

if not merged_files:
    raise FileNotFoundError(f"No merged CSV found in {clean_dir}. "
                            "Check the filename or generate the merged dataset.")

data_path = merged_files[0]  # use the first match
output_dir = project_root / 'outputs' / 'analysis'

print(f"Project root: {project_root}")
print(f"Using merged file: {data_path.name}")
print(f"Output dir: {output_dir}")


Project root: C:\Users\gdlew\OneDrive\MONASH\Yr 3 Sem 1\ECC3479 Data and Evidence in economics\ecc3479-project
Using merged file: merged_2016_2021.csv
Output dir: C:\Users\gdlew\OneDrive\MONASH\Yr 3 Sem 1\ECC3479 Data and Evidence in economics\ecc3479-project\outputs\analysis


## Load Cleaned Census Data

In [24]:
# Load the cleaned merged dataset
df = pd.read_csv(data_path)

# Inspect the data
print("Dataset shape:", df.shape)
print("\nColumns:", list(df.columns))
print("\nFirst 5 rows:")
display(df.head())

print("\nSummary statistics:")
display(df.describe(include='all'))

# Filter to analytic sample
import re

# Harmonise income brackets
def harmonise_income_bracket(bracket: str) -> str:
    if bracket in {
        "$3,000-$3,499 ($156,000-$181,999)",
        "$3,500 Or More ($182,000 Or More)",
    }:
        return "$3,000 Or More ($156,000 Or More)"
    return bracket

def parse_weekly_midpoint(bracket: str) -> float:
    if pd.isna(bracket):
        return np.nan

    label = str(bracket).strip()
    weekly_label = label.split(" (")[0]

    if weekly_label == "Negative Income":
        return -75.0
    if weekly_label == "Nil Income":
        return 0.0
    if weekly_label == "Not Stated":
        return np.nan

    if "Or More" in weekly_label:
        lower_match = re.search(r"\$([\d,]+)", weekly_label)
        if lower_match:
            lower = float(lower_match.group(1).replace(",", ""))
            return lower + 250.0
        return np.nan

    range_match = re.findall(r"\$([\d,]+)", weekly_label)
    if len(range_match) >= 2:
        lower = float(range_match[0].replace(",", ""))
        upper = float(range_match[1].replace(",", ""))
        return (lower + upper) / 2

    return np.nan

# Apply processing
df["harmonised_income_bracket"] = df["income_bracket"].map(harmonise_income_bracket)
df["income_midpoint"] = df["income_bracket"].map(parse_weekly_midpoint)
df["harmonised_midpoint"] = df["harmonised_income_bracket"].map(parse_weekly_midpoint)

EXCLUDED_EDUCATION = {
    "Supplementary Codes",
    "Not stated",
    "Not applicable",
    "Total",
}

ANALYTIC_EDUCATION_ORDER = [
    "Secondary Education - Years 9 and below",
    "Secondary Education - Years 10 and above",
    "Certificate I & II Level",
    "Certificate III & IV Level",
    "Advanced Diploma and Diploma Level",
    "Bachelor Degree Level",
    "Graduate Diploma and Graduate Certificate Level",
    "Postgraduate Degree Level",
]

df["is_analytic_education"] = ~df["education"].isin(EXCLUDED_EDUCATION)
df["education_rank"] = df["education"].map(
    {education: rank + 1 for rank, education in enumerate(ANALYTIC_EDUCATION_ORDER)}
)
df["is_stated_income"] = df["harmonised_income_bracket"] != "Not Stated"
df["weighted_income"] = df["harmonised_midpoint"] * df["count"]

# Filter to analytic sample
analytic_df = df[df['is_analytic_education'] & df['is_stated_income']].copy()
print(f"\nAnalytic sample shape: {analytic_df.shape}")

# Education rank mapping (from EDA)
education_rank_map = {
    "Secondary Education - Years 9 and below": 1,
    "Secondary Education - Years 10 and above": 2,
    "Certificate I & II Level": 3,
    "Certificate III & IV Level": 4,
    "Advanced Diploma and Diploma Level": 5,
    "Bachelor Degree Level": 6,
    "Graduate Diploma and Graduate Certificate Level": 7,
    "Postgraduate Degree Level": 8,
}

analytic_df['education_rank'] = analytic_df['education'].map(education_rank_map)

print("\nEducation ranks assigned:")
display(analytic_df[['education', 'education_rank']].drop_duplicates().sort_values('education_rank'))

Dataset shape: (396, 4)

Columns: ['year', 'income_bracket', 'education', 'count']

First 5 rows:


,year,income_bracket,education,count
0,2016,Negative Income,Postgraduate Degree Level,1503
1,2016,Nil Income,Postgraduate Degree Level,16045
2,2016,"$1-$149 ($1-$7,799)",Postgraduate Degree Level,5306
3,2016,"$150-$299 ($7,800-$15,599)",Postgraduate Degree Level,10087
4,2016,"$300-$399 ($15,600-$20,799)",Postgraduate Degree Level,11968



Summary statistics:


,year,income_bracket,education,count
count,396.000000,396,396,396.000000
unique,NaN,18,12,NaN
top,NaN,Negative Income,Postgraduate Degree Level,NaN
freq,NaN,24,33,NaN
mean,2018.575758,NaN,NaN,98318.787879
std,2.502013,NaN,NaN,180183.420042
min,2016.000000,NaN,NaN,0.000000
25%,2016.000000,NaN,NaN,5583.250000
50%,2021.000000,NaN,NaN,25573.000000
75%,2021.000000,NaN,NaN,94917.000000



Analytic sample shape: (248, 11)

Education ranks assigned:


,education,education_rank
112,Secondary Education - Years 9 and below,1
80,Secondary Education - Years 10 and above,2
96,Certificate I & II Level,3
64,Certificate III & IV Level,4
48,Advanced Diploma and Diploma Level,5
32,Bachelor Degree Level,6
16,Graduate Diploma and Graduate Certificate Level,7
0,Postgraduate Degree Level,8


## Analysis Ambition

This analysis is **descriptive**, aiming to quantify the conditional correlations between education level and income in the Australian population using 2016 and 2021 census data. We do not attempt to identify a causal treatment effect of education on income, as education attainment is not randomly assigned and we lack experimental or quasi-experimental variation. Instead, we estimate predictive associations while controlling for observable factors like time period.

## Econometric Specification

We estimate the following weighted least squares (WLS) regression to examine the relationship between education and income:

**income_midpoint = β₀ + β₁ × education_rank + β₂ × year_2021 + ε**

Where:
- **income_midpoint**: The midpoint value of the income bracket in Australian dollars per week
- **education_rank**: An ordinal variable ranking education levels from 1 (lowest: Secondary Education - Years 9 and below) to 8 (highest: Postgraduate Degree Level)
- **year_2021**: A dummy variable equal to 1 for 2021 observations, 0 for 2016
- **ε**: Error term

**Functional form**: Linear OLS with ordinal education variable.

**Regressors**: Education rank and year dummy.

**Sample**: All observations with stated income and analytic education categories (excluding supplementary codes, not stated, not applicable, and total categories).

**Error structure**: We use weighted least squares with weights equal to the count of individuals in each cell, assuming heteroskedasticity related to cell size. This gives proper representation to larger population groups.

**Justification**: 
- Education rank captures the ordinal nature of education levels better than dummy variables.
- Year dummy controls for aggregate changes in income distribution between 2016 and 2021.
- WLS ensures the regression reflects the population distribution rather than treating each cell equally.

In [25]:
# Prepare regression data
# Group by year and education to get weighted averages
reg_df = analytic_df.groupby(['year', 'education'], observed=True).agg(
    weighted_mean_income=('weighted_income', 'sum'),
    total_count=('count', 'sum'),
    education_rank=('education_rank', 'first')
).reset_index()

reg_df['weighted_mean_income'] /= reg_df['total_count']
reg_df['year_2021'] = (reg_df['year'] == 2021).astype(int)

print("Regression dataset:")
display(reg_df.head(10))

# Define variables
y = reg_df['weighted_mean_income']
X = reg_df[['education_rank', 'year_2021']]
X = sm.add_constant(X)  # Add intercept
weights = reg_df['total_count']

# Run WLS regression
model = sm.WLS(y, X, weights=weights)
results = model.fit()

print("\nRegression Results:")
print(results.summary())

Regression dataset:


,year,education,weighted_mean_income,total_count,education_rank,year_2021
0,2016,Advanced Diploma and Diploma Level,1276.040174,701942,5,0
1,2016,Bachelor Degree Level,1534.662148,1236667,6,0
2,2016,Certificate I & II Level,635.463548,6049,3,0
3,2016,Certificate III & IV Level,1090.839594,1988369,4,0
4,2016,Graduate Diploma and Graduate Certificate Level,1695.063977,133704,7,0
5,2016,Postgraduate Degree Level,1755.015693,459793,8,0
6,2016,Secondary Education - Years 10 and above,739.152290,2699216,2,0
7,2016,Secondary Education - Years 9 and below,487.011618,674970,1,0
8,2021,Advanced Diploma and Diploma Level,1406.936423,817960,5,1
9,2021,Bachelor Degree Level,1704.554532,1552228,6,1



Regression Results:
                             WLS Regression Results                             
Dep. Variable:     weighted_mean_income   R-squared:                       0.983
Model:                              WLS   Adj. R-squared:                  0.980
Method:                   Least Squares   F-statistic:                     365.7
Date:                  Tue, 05 May 2026   Prob (F-statistic):           3.75e-12
Time:                          23:13:40   Log-Likelihood:                -92.876
No. Observations:                    16   AIC:                             191.8
Df Residuals:                        13   BIC:                             194.1
Df Model:                             2                                         
Covariance Type:              nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
con

## Regression Table

In [26]:
# Create a formatted regression table
import pandas as pd

# Extract coefficients and standard errors
reg_table = pd.DataFrame({
    'Coefficient': results.params,
    'Std. Error': results.bse,
    't-statistic': results.tvalues,
    'P-value': results.pvalues
})

# Format the table for display
reg_table_display = pd.DataFrame({
    'Variable': ['Constant', 'Education Rank', 'Year 2021'],
    'Coefficient': [
        f"{results.params['const']:.4f}",
        f"{results.params['education_rank']:.4f}",
        f"{results.params['year_2021']:.4f}"
    ],
    'Std. Error': [
        f"({results.bse['const']:.4f})",
        f"({results.bse['education_rank']:.4f})",
        f"({results.bse['year_2021']:.4f})"
    ],
    'Significance': [
        '***' if results.pvalues['const'] < 0.001 else '**' if results.pvalues['const'] < 0.01 else '*' if results.pvalues['const'] < 0.05 else '',
        '***' if results.pvalues['education_rank'] < 0.001 else '**' if results.pvalues['education_rank'] < 0.01 else '*' if results.pvalues['education_rank'] < 0.05 else '',
        '***' if results.pvalues['year_2021'] < 0.001 else '**' if results.pvalues['year_2021'] < 0.01 else '*' if results.pvalues['year_2021'] < 0.05 else ''
    ]
})

print("=" * 75)
print("TABLE 1: WEIGHTED LEAST SQUARES REGRESSION")
print("Dependent Variable: Weekly Income Midpoint (AUD)")
print("=" * 75)
display(reg_table_display)
print("=" * 75)
print(f"Observations: {int(results.nobs)}")
print(f"R-squared: {results.rsquared:.4f}")
print(f"Adjusted R-squared: {results.rsquared_adj:.4f}")
print(f"F-statistic: {results.fvalue:.4f} ({results.f_pvalue:.2e})")
print("=" * 75)
print("Note: *** p<0.001, ** p<0.01, * p<0.05")
print("Standard errors in parentheses. Weights: cell population counts.")

TABLE 1: WEIGHTED LEAST SQUARES REGRESSION
Dependent Variable: Weekly Income Midpoint (AUD)


,Variable,Coefficient,Std. Error,Significance
0,Constant,319.9810,(35.0402),***
1,Education Rank,195.8755,(7.3815),***
2,Year 2021,120.7521,(29.6192),**


Observations: 16
R-squared: 0.9825
Adjusted R-squared: 0.9798
F-statistic: 365.7009 (3.75e-12)
Note: *** p<0.001, ** p<0.01, * p<0.05
Standard errors in parentheses. Weights: cell population counts.


## Interpretation of Main Coefficients

The coefficient on **education_rank** is positive and statistically significant at conventional levels. A one-unit increase in education rank (corresponding to moving up one education level, e.g., from "Advanced Diploma" to "Bachelor Degree") is associated with an increase of approximately $196 in weekly income, holding the year constant.

The coefficient on **year_2021** is also positive and significant, indicating that incomes were higher in 2021 compared to 2016, holding education level constant. This represents an increase of about $121 in weekly income between the two census years.

The constant term represents the predicted income for the lowest education level (rank 1) in 2016.

All coefficients are in Australian dollars per week, and the interpretation holds other variables in the model constant.

## Threats to Validity

As a descriptive analysis, this study does not claim to identify causal effects. The main limitations are:

1. **Omitted Variables**: Unobserved factors such as innate ability, family socioeconomic background, geographic location, and work experience are likely correlated with both education attainment and income, biasing the education coefficient upward.

2. **Selection Bias**: Individuals self-select into education levels based on expected returns, preferences, and constraints. This selection is not random and may confound the observed associations.

3. **Measurement Error**: Income is measured in brackets rather than exact values, and education categories may not perfectly capture skill differences. This could attenuate the estimated coefficients.

4. **Sample Selection**: The analysis excludes individuals with unstated income or non-analytic education categories, which may not be random.

These limitations mean the estimated associations likely overstate the true predictive relationship between education and income due to confounding factors.

## Reproducibility and Pipeline Check

This notebook uses the cleaned data from `data/clean/merged_2016_2021.csv`, which is produced by the data cleaning scripts in `src/`. The full pipeline is:

1. Raw data → `src/01_load_2016_data.py`, `src/02_clean_2016_data.py`, etc.
2. Clean data → `src/03_merge_data_sets.py` 
3. Analysis → This notebook

To reproduce:
- Ensure all dependencies in `requirements.txt` are installed
- Run the cleaning scripts in order
- Execute this notebook

**Updated files**: Added `requirements.txt` (statsmodels), created `outputs/analysis/analysis.ipynb`, and updated `README.md` with analysis instructions.

In [27]:
# Verify pipeline components exist
pipeline_files = [
    'README.md',
    'requirements.txt',
    'data/clean/merged_2016_2021.csv',
    'src/01_load_2016_data.py',
    'src/02_clean_2016_data.py',
    'src/03_merge_data_sets.py',
    'src/04_eda.py',
]

print("Pipeline verification:")
for file in pipeline_files:
    exists = (project_root / file).exists()
    print(f"✓ {file}: {'EXISTS' if exists else 'MISSING'}")

print(f"\nNotebook saved at: {output_dir / 'analysis.ipynb'}")

Pipeline verification:
✓ README.md: EXISTS
✓ requirements.txt: EXISTS
✓ data/clean/merged_2016_2021.csv: EXISTS
✓ src/01_load_2016_data.py: EXISTS
✓ src/02_clean_2016_data.py: EXISTS
✓ src/03_merge_data_sets.py: EXISTS
✓ src/04_eda.py: EXISTS

Notebook saved at: C:\Users\gdlew\OneDrive\MONASH\Yr 3 Sem 1\ECC3479 Data and Evidence in economics\ecc3479-project\outputs\analysis\analysis.ipynb
